In [6]:
from time import time
import pandas as pd
import re
import os
import numpy as np
from scipy.signal import medfilt, savgol_filter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [9]:
# Read the CSV into a DataFrame
log_dir_path = '/data/ros2/ros2_ws2/arm_bot_pos_control/src/scripts/logs'
dataset_dir_path = '/data/ros2/ros2_ws2/arm_bot_pos_control/src/scripts/Joint_states'
new_dataset_dir_path = '/data/ros2/ros2_ws2/arm_bot_pos_control/src/scripts/new_generatedDataset1'
new_dataset_plot_path = '/data/ros2/ros2_ws2/arm_bot_pos_control/src/scripts/new_generatedDataset1/plots'


def smart_filter(x):
    # remove spikes
    x1 = medfilt(x, kernel_size=5)

    # smooth curve while preserving shape
    x2 = savgol_filter(x1, window_length=11, polyorder=3)

    return x2


for filename in os.listdir(log_dir_path):
    match = re.search(r'path_(\d+)_log_(\d+)\.csv$', filename)
    if match and int(match.group(1)) >= 614 and int(match.group(1)) <= 614:
        log_file_path = os.path.join(log_dir_path, filename)
        print(f'Processing log file: {filename}')

        df = pd.read_csv(log_file_path)

        time = df['time_elapsed'].tolist()
        pos1 = df['pos1'].tolist()
        pos2 = df['pos2'].tolist()
        pos3 = df['pos3'].tolist()
        vel1 = df['vel1'].tolist()
        vel2 = df['vel2'].tolist()
        vel3 = df['vel3'].tolist()
        tau1 = df['torque1'].tolist()
        tau2 = df['torque2'].tolist()
        tau3 = df['torque3'].tolist()

        time, pos1, pos2, pos3, vel1, vel2, vel3, tau1, tau2, tau3 = map(smart_filter, [time, pos1, pos2, pos3, vel1, vel2, vel3, tau1, tau2, tau3])
        
        df1 = pd.DataFrame({
            'time_elapsed': time,
            'pos1': pos1,
            'pos2': pos2,
            'pos3': pos3,
            'vel1': vel1,
            'vel2': vel2,
            'vel3': vel3,
            'torque1': tau1,
            'torque2': tau2,
            'torque3': tau3
        })

        # add acceleration fields by calculating the difference between consecutive velocity values and filling the first value with 0.
        df1['da1'] = df1['vel1'].diff().fillna(0).round(6)  # da1
        df1['da2'] = df1['vel2'].diff().fillna(0).round(6)  # da2
        df1['da3'] = df1['vel3'].diff().fillna(0).round(6)  # da3

        # Rename the columns to match your desired header names
        df1 = df1.rename(columns={
            'time_elapsed': 't',
            'pos1': 'dp1',
            'pos2': 'dp2',
            'pos3': 'dp3',
            'vel1': 'dv1',
            'vel2': 'dv2',
            'vel3': 'dv3',
            'torque1': 'tau1',
            'torque2': 'tau2',
            'torque3': 'tau3',
            'da1': 'da1',
            'da2': 'da2',
            'da3': 'da3'
        })

    # plot df and df1 to compare the original and processed data
        fig, axs = plt.subplots(3, 3, figsize=(15, 10))
        axs[0, 0].plot(df['time_elapsed'].to_numpy(), df['pos1'].to_numpy(), label='Original pos1')
        axs[0, 0].plot(df1['t'].to_numpy(), df1['dp1'].to_numpy(), label='Processed pos1')
        axs[0, 0].legend()
        axs[0, 0].set_title('Position 1')
        axs[0, 1].plot(df['time_elapsed'].to_numpy(), df['pos2'].to_numpy(), label='Original pos2')
        axs[0, 1].plot(df1['t'].to_numpy(), df1['dp2'].to_numpy(), label='Processed pos2')
        axs[0, 1].legend()
        axs[0, 1].set_title('Position 2')
        axs[0, 2].plot(df['time_elapsed'].to_numpy(), df['pos3'].to_numpy(), label='Original pos3')
        axs[0, 2].plot(df1['t'].to_numpy(), df1['dp3'].to_numpy(), label='Processed pos3')
        axs[0, 2].legend()
        axs[0, 2].set_title('Position 3')
        axs[1, 0].plot(df['time_elapsed'].to_numpy(), df['vel1'].to_numpy(), label='Original vel1')
        axs[1, 0].plot(df1['t'].to_numpy(), df1['dv1'].to_numpy(), label='Processed vel1')
        axs[1, 0].legend()
        axs[1, 0].set_title('Velocity 1')
        axs[1, 1].plot(df['time_elapsed'].to_numpy(), df['vel2'].to_numpy(), label='Original vel2')
        axs[1, 1].plot(df1['t'].to_numpy(), df1['dv2'].to_numpy(), label='Processed vel2')
        axs[1, 1].legend()
        axs[1, 1].set_title('Velocity 2')
        axs[1, 2].plot(df['time_elapsed'].to_numpy(), df['vel3'].to_numpy(), label='Original vel3')
        axs[1, 2].plot(df1['t'].to_numpy(), df1['dv3'].to_numpy(), label='Processed vel3')
        axs[1, 2].legend()
        axs[1, 2].set_title('Velocity 3')
        axs[2, 0].plot(df['time_elapsed'].to_numpy(), df['torque1'].to_numpy(), label='Original torque1')
        axs[2, 0].plot(df1['t'].to_numpy(), df1['tau1'].to_numpy(), label='Processed torque1')
        axs[2, 0].legend()
        axs[2, 0].set_title('Torque 1')
        axs[2, 1].plot(df['time_elapsed'].to_numpy(), df['torque2'].to_numpy(), label='Original torque2')
        axs[2, 1].plot(df1['t'].to_numpy(), df1['tau2'].to_numpy(), label='Processed torque2')
        axs[2, 1].legend()
        axs[2, 1].set_title('Torque 2')
        axs[2, 2].plot(df['time_elapsed'].to_numpy(), df['torque3'].to_numpy(), label='Original torque3')
        axs[2, 2].plot(df1['t'].to_numpy(), df1['tau3'].to_numpy(), label='Processed torque3')
        axs[2, 2].legend()
        axs[2, 2].set_title('Torque 3')
        plt.tight_layout()

        # instead of showing the plot, save it to a file
        plot_file_path = os.path.join(new_dataset_plot_path, f'path_{match.group(1)}_plot_{match.group(2)}.png')
        print(f'Saving plot to: {plot_file_path}')
        plt.savefig(plot_file_path)
        # plt.show()


        df1 = df1.T
        output_file_path = os.path.join(new_dataset_dir_path, f'path_{match.group(1)}_joint_states_{match.group(2)}.csv')
        print(f'Saving processed data to: {output_file_path}')
        df1.to_csv(output_file_path, header=False, index=True)
        df1.head()




Processing log file: path_614_log_1.csv
Saving plot to: /data/ros2/ros2_ws2/arm_bot_pos_control/src/scripts/new_generatedDataset1/plots/path_614_plot_1.png
Saving processed data to: /data/ros2/ros2_ws2/arm_bot_pos_control/src/scripts/new_generatedDataset1/path_614_joint_states_1.csv


In [17]:
import pandas as pd
import re
import os

dataset_dir_path = "/data/ros2/ros2_ws2/arm_bot_pos_control/src/scripts/Joint_states"

print(f'Processing dataset directory: {dataset_dir_path}')
count = 0
for filename in os.listdir(dataset_dir_path):
    print(f'Checking file: {filename}')
    match = re.search(r'path_(\d+)_joint_states.csv$', filename)
    if match and int(match.group(1)) <= 450 and int(match.group(1)) >= 81:
        log_file_path = os.path.join(dataset_dir_path, filename)
        print(f'Processing log file: {filename}')

        df = pd.read_csv(log_file_path, index_col=0)

        # Extract relevant arrays by row index, convert to numpy arrays
        pos1 = df.loc['dp1'].values
        pos2 = df.loc['dp2'].values
        pos3 = df.loc['dp3'].values

        # convert the position values from radians to degrees
    # if {float(max(pos1)* 180 / 3.141592653589793) > 20 or float(min(pos1)* 180 / 3.141592653589793) < -20}:
        print(f'joint 1 Max position: {max(pos1)* 180 / 3.141592653589793}, Min position: {min(pos1)* 180 / 3.141592653589793}')
        print(f'joint 2 Max position: {max(pos2)* 180 / 3.141592653589793}, Min position: {min(pos2)* 180 / 3.141592653589793}')
        print(f'joint 3 Max position: {max(pos3)* 180 / 3.141592653589793}, Min position: {min(pos3)* 180 / 3.141592653589793}')
        count += 1
    # else:
    #     print(f'Skipping file: {filename} - does not match pattern or path ID out of range')
print(f'Total files with joint 1 position exceeding ±20 degrees: {count}')

Processing dataset directory: /data/ros2/ros2_ws2/arm_bot_pos_control/src/scripts/Joint_states
Checking file: path_741_joint_states.csv
Checking file: path_1048_joint_states.csv
Checking file: path_783_joint_states.csv
Checking file: path_734_joint_states.csv
Checking file: path_995_joint_states.csv
Checking file: path_829_joint_states.csv
Checking file: path_919_joint_states.csv
Checking file: path_671_joint_states.csv
Checking file: path_786_joint_states.csv
Checking file: path_839_joint_states.csv
Checking file: path_603_joint_states.csv
Checking file: path_1001_joint_states.csv
Checking file: path_1092_joint_states.csv
Checking file: path_821_joint_states.csv
Checking file: path_1111_joint_states.csv
Checking file: path_870_joint_states.csv
Checking file: path_787_joint_states.csv
Checking file: path_1064_joint_states.csv
Checking file: path_637_joint_states.csv
Checking file: path_680_joint_states.csv
Checking file: path_894_joint_states.csv
Checking file: path_1003_joint_states.c